Self-attention is a mechanism that lets each token in a sequence look at every other token and decide how much to focus on each one.

For every token, three vectors are created — a **Query** (what it's looking for), a **Key** (what it advertises about itself), and a **Value** (what it actually shares). The query of one token is compared against the keys of all other tokens using a dot product, which produces a score representing relevance. These scores are scaled by √d_k to keep them numerically stable, then passed through softmax to turn them into probabilities that sum to 1. Finally, those probabilities are used to take a weighted sum of all the value vectors, producing a new context-aware representation for that token.

The beauty of it is that every token does this simultaneously in parallel, and any token can directly attend to any other token regardless of distance — solving the long-range dependency problem that plagued older RNN-based models. Running multiple attention heads in parallel lets the model capture different types of relationships at once, like syntax, coreference, and semantics, all in a single layer.

In [1]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (..., n, d_k)
    K: (..., n, d_k)
    V: (..., n, d_v)
    mask: (..., n, n)  True = blocked position
    Returns: output (..., n, d_v), weights (..., n, n)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)   # (..., n, n)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    weights = softmax(scores, axis=-1)                 # (..., n, n)
    output  = weights @ V                              # (..., n, d_v)
    return output, weights


class SelfAttention:
    """Single-head self-attention layer (NumPy)."""

    def __init__(self, d_model, d_k, d_v):
        scale = np.sqrt(2.0 / (d_model + d_k))
        self.W_Q = np.random.randn(d_model, d_k) * scale
        self.W_K = np.random.randn(d_model, d_k) * scale
        self.W_V = np.random.randn(d_model, d_v) * scale

    def forward(self, X, mask=None):
        """X: (n, d_model)  →  output: (n, d_v)"""
        Q = X @ self.W_Q
        K = X @ self.W_K
        V = X @ self.W_V
        return scaled_dot_product_attention(Q, K, V, mask)


# ── Quick test ─────────────────────────────────────────────
n, d_model, d_k, d_v = 6, 32, 16, 16
X   = np.random.randn(n, d_model)
attn = SelfAttention(d_model, d_k, d_v)
out, weights = attn.forward(X)
print("Output :", out.shape)     # (6, 16)
print("Weights:", weights.shape) # (6, 6)
print("Row sum :", weights[0].sum().round(4))  # 1.0

Output : (6, 16)
Weights: (6, 6)
Row sum : 1.0


In [2]:
class MultiHeadAttention:
    """Multi-head self-attention (NumPy, no training)."""

    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.h   = num_heads
        self.d_k = d_model // num_heads
        scale    = 0.1

        # Pack all heads into single matrices for efficiency
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale

    def _split_heads(self, x):
        """(n, d_model) → (h, n, d_k)"""
        n = x.shape[0]
        x = x.reshape(n, self.h, self.d_k)
        return x.transpose(1, 0, 2)          # (h, n, d_k)

    def _merge_heads(self, x):
        """(h, n, d_k) → (n, d_model)"""
        h, n, d_k = x.shape
        return x.transpose(1, 0, 2).reshape(n, h * d_k)

    def forward(self, X, mask=None):
        Q = self._split_heads(X @ self.W_Q)  # (h, n, d_k)
        K = self._split_heads(X @ self.W_K)
        V = self._split_heads(X @ self.W_V)

        out, weights = scaled_dot_product_attention(Q, K, V, mask)
        merged = self._merge_heads(out)       # (n, d_model)
        return merged @ self.W_O, weights     # (n, d_model)


mha  = MultiHeadAttention(d_model=64, num_heads=8)
X    = np.random.randn(10, 64)
out, w = mha.forward(X)
print(out.shape, w.shape)  # (10, 64)  (8, 10, 10)

(10, 64) (8, 10, 10)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        d_k    = Q.size(-1)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # (..., n, n)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = self.dropout(F.softmax(scores, dim=-1))
        return weights @ V, weights


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.attn    = ScaledDotProductAttention(dropout)
        self.dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        for module in [self.W_Q, self.W_K, self.W_V, self.W_O]:
            nn.init.xavier_uniform_(module.weight)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, n, d_model) → (B, h, n, d_k)"""
        B, n, _ = x.shape
        return x.view(B, n, self.num_heads, self.d_k).transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, h, n, d_k) → (B, n, d_model)"""
        B, h, n, d_k = x.shape
        return x.transpose(1, 2).contiguous().view(B, n, h * d_k)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        Q = self._split_heads(self.W_Q(x))   # (B, h, n, d_k)
        K = self._split_heads(self.W_K(x))
        V = self._split_heads(self.W_V(x))

        context, weights = self.attn(Q, K, V, mask)  # (B, h, n, d_k)
        out = self.W_O(self._merge_heads(context))    # (B, n, d_model)
        return out, weights


# ── Full Transformer Block ──────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int,
                 ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LN + residual (modern style)
        attn_out, weights = self.attn(self.ln1(x), mask)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x, weights

In [4]:
class CausalSelfAttention(nn.Module):
    """Masked self-attention for autoregressive language models."""

    def __init__(self, d_model: int, num_heads: int,
                 max_seq_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.mha = MultiHeadSelfAttention(d_model, num_heads, dropout)

        # Pre-compute causal mask and register as buffer (not a parameter)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer('causal_mask', mask)

    def forward(self, x: torch.Tensor):
        n = x.size(1)
        mask = self.causal_mask[:n, :n]                   # trim to current length
        mask = mask.unsqueeze(0).unsqueeze(0)             # (1, 1, n, n)
        return self.mha(x, mask)


# ── Test ───────────────────────────────────────────────────
B, n, d = 2, 16, 128
x   = torch.randn(B, n, d)
csa = CausalSelfAttention(d_model=d, num_heads=4, max_seq_len=64)
out, w = csa(x)
print(out.shape)   # (2, 16, 128)
print(w.shape)     # (2, 4, 16, 16)

# Verify causal: upper triangle of weights should be ~0
print("Upper triangle near zero:", w[0, 0].triu(1).abs().max().item() < 1e-6)

torch.Size([2, 16, 128])
torch.Size([2, 4, 16, 16])
Upper triangle near zero: True


In [5]:
# PyTorch 2.0+ has a built-in Flash Attention — use it in production
# It's IO-aware: never materializes the full n×n matrix in HBM

class EfficientAttention(nn.Module):
    """Uses torch.nn.functional.scaled_dot_product_attention (Flash Attention)."""

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x, is_causal=True):
        B, n, d = x.shape
        h, d_k = self.num_heads, self.d_k

        Q = self.W_Q(x).view(B, n, h, d_k).transpose(1, 2)  # (B, h, n, d_k)
        K = self.W_K(x).view(B, n, h, d_k).transpose(1, 2)
        V = self.W_V(x).view(B, n, h, d_k).transpose(1, 2)

        # Flash Attention — O(n) memory, same result as standard attention
        out = F.scaled_dot_product_attention(
            Q, K, V,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal
        )

        out = out.transpose(1, 2).contiguous().view(B, n, d)
        return self.W_O(out)


model = EfficientAttention(d_model=256, num_heads=8)
x = torch.randn(4, 512, 256)
print(model(x).shape)   # (4, 512, 256)

torch.Size([4, 512, 256])


In [6]:
class CachedAttention(nn.Module):
    """Self-attention with KV cache for efficient token-by-token generation."""

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        # KV cache (populated during generation)
        self.cache_K: torch.Tensor | None = None
        self.cache_V: torch.Tensor | None = None

    def reset_cache(self):
        self.cache_K = self.cache_V = None

    def forward(self, x: torch.Tensor, use_cache: bool = False):
        B, n, d = x.shape
        h, d_k  = self.num_heads, self.d_k

        def split(t):
            return t.view(B, -1, h, d_k).transpose(1, 2)

        Q = split(self.W_Q(x))        # new query
        K = split(self.W_K(x))        # new keys
        V = split(self.W_V(x))        # new values

        if use_cache:
            # Append to cache
            if self.cache_K is not None:
                K = torch.cat([self.cache_K, K], dim=2)
                V = torch.cat([self.cache_V, V], dim=2)
            self.cache_K, self.cache_V = K, V

        # Compute attention (Q attends to all cached K, V)
        out = F.scaled_dot_product_attention(Q, K, V, is_causal=(not use_cache))
        out = out.transpose(1, 2).contiguous().view(B, n, d)
        return self.W_O(out)